In [1]:
import pandas as pd
import numpy as np

features = pd.read_csv("../outputs/feature_table.csv")

print("Dataset shape:", features.shape)
print("Columns:", list(features.columns))
features.head()

Dataset shape: (95, 33)
Columns: ['Skill', 'Freq', 'Trend_Slope', 'Recency_Weight', 'Upcoming_Flag', 'Is_Taught_MSRIT', 'Taught_Institute_Count', 'Institutional_Gap_Score', 'Company_Spread', 'Company_Spread_Norm', 'Velocity', 'Volatility', 'Semantic_Centrality', 'Semantic_Gap_Distance', 'NLP_Rarity_Score', 'Skill_Cluster', 'Cluster_0', 'Cluster_1', 'Cluster_2', 'Cluster_3', 'Cluster_4', 'Cluster_5', 'Cluster_6', 'Cluster_7', 'Skill_Domain', 'Domain_AI/ML', 'Domain_Cloud', 'Domain_Data', 'Domain_DevOps', 'Domain_Other', 'Domain_Security', 'Domain_Systems/Infra', 'Domain_Web/Mobile']


,Skill,Freq,Trend_Slope,Recency_Weight,Upcoming_Flag,Is_Taught_MSRIT,Taught_Institute_Count,Institutional_Gap_Score,Company_Spread,Company_Spread_Norm,...,Cluster_7,Skill_Domain,Domain_AI/ML,Domain_Cloud,Domain_Data,Domain_DevOps,Domain_Other,Domain_Security,Domain_Systems/Infra,Domain_Web/Mobile
0,advanced sql,8,-4.000000,0.812500,1,1,3,0.4,8,0.8,...,False,Data,False,False,True,False,False,False,False,False
1,api gateway,1,0.000000,0.250000,0,1,1,0.8,1,0.1,...,False,Other,False,False,False,False,True,False,False,False
2,api rate limiting,1,0.000000,1.000000,0,0,0,1.0,1,0.1,...,False,Other,False,False,False,False,True,False,False,False
3,auto scaling,1,0.000000,0.500000,0,0,0,1.0,1,0.1,...,False,Other,False,False,False,False,True,False,False,False
4,aws,29,-1.571429,0.318966,0,0,0,1.0,10,1.0,...,False,Cloud,False,True,False,False,False,False,False,False


In [2]:
features["Demand_Score"] = (
    features["Freq"]                 * 0.25 +
    features["Trend_Slope"]          * 0.20 +
    features["Recency_Weight"]       * 0.20 +
    features["Velocity"]             * 0.15 +
    features["Company_Spread_Norm"]  * 0.20
).round(4)

threshold = features["Demand_Score"].quantile(0.65)
print(f"Demand_Score threshold (65th percentile): {threshold:.4f}")

features["Label_MSRIT"] = (
    (features["Demand_Score"] > threshold) &
    (features["Is_Taught_MSRIT"] == 0)
).astype(int)

print("Class distribution:")
print(features["Label_MSRIT"].value_counts())
print(f"Imbalance ratio: {features['Label_MSRIT'].value_counts()[0] / features['Label_MSRIT'].value_counts()[1]:.1f}:1")
print()
print("Demand_Score stats:")
print(features["Demand_Score"].describe().round(3))

Demand_Score threshold (65th percentile): 1.6223
Class distribution:
Label_MSRIT
0    83
1    12
Name: count, dtype: int64
Imbalance ratio: 6.9:1

Demand_Score stats:
count    95.000
mean      1.859
std       2.386
min       0.270
25%       0.370
50%       0.470
75%       2.717
max      12.074
Name: Demand_Score, dtype: float64


In [3]:
from sklearn.model_selection import train_test_split

FEATURE_COLS = [
    "Freq",
    "Trend_Slope",
    "Recency_Weight",
    "Upcoming_Flag",
    "Taught_Institute_Count",
    "Company_Spread",
    "Company_Spread_Norm",
    "Velocity",
    "Volatility",
    "Institutional_Gap_Score",
]

missing_cols = [c for c in FEATURE_COLS if c not in features.columns]
if missing_cols:
    print("WARNING - missing columns:", missing_cols)
else:
    print("All feature columns present")

X = features[FEATURE_COLS]
y = features["Label_MSRIT"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print("Train size:", X_train.shape[0], "| Test size:", X_test.shape[0])
print("Train class dist:", y_train.value_counts().to_dict())
print("Test class dist:", y_test.value_counts().to_dict())

All feature columns present
Train size: 71 | Test size: 24
Train class dist: {0: 62, 1: 9}
Test class dist: {0: 21, 1: 3}


In [ ]:
NLP_CANDIDATE_COLS = [
    "Semantic_Centrality",
    "Semantic_Gap_Distance",
    "NLP_Rarity_Score",
]

NLP_CANDIDATE_COLS += [c for c in features.columns
                        if c.startswith("Cluster_") or c.startswith("Domain_")]

NLP_FEATURE_COLS = [c for c in NLP_CANDIDATE_COLS if c in features.columns]
print(f"{len(NLP_FEATURE_COLS)} NLP features found: {NLP_FEATURE_COLS}")

FEATURE_COLS_EXTENDED = FEATURE_COLS + NLP_FEATURE_COLS
missing_ext = [c for c in FEATURE_COLS_EXTENDED if c not in features.columns]
if missing_ext:
    print(f"WARNING: columns missing from CSV, dropping: {missing_ext}")
    FEATURE_COLS_EXTENDED = [c for c in FEATURE_COLS_EXTENDED if c in features.columns]

print(f"Final feature set: {len(FEATURE_COLS_EXTENDED)} features")

X_nlp = features[FEATURE_COLS_EXTENDED]
y_nlp = features["Label_MSRIT"]

from sklearn.model_selection import train_test_split
X_train_nlp, X_test_nlp, y_train_nlp, y_test_nlp = train_test_split(
    X_nlp, y_nlp, test_size=0.25, random_state=42, stratify=y_nlp
)
print("NLP-extended train size:", X_train_nlp.shape[0], "| Test size:", X_test_nlp.shape[0])


19 NLP features found: ['Semantic_Centrality', 'Semantic_Gap_Distance', 'NLP_Rarity_Score', 'Cluster_0', 'Cluster_1', 'Cluster_2', 'Cluster_3', 'Cluster_4', 'Cluster_5', 'Cluster_6', 'Cluster_7', 'Domain_AI/ML', 'Domain_Cloud', 'Domain_Data', 'Domain_DevOps', 'Domain_Other', 'Domain_Security', 'Domain_Systems/Infra', 'Domain_Web/Mobile']
Final feature set: 29 features
NLP-extended train size: 71 | Test size: 24


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import cross_val_predict, StratifiedKFold

def eval_model(X_tr, y_tr, X_te, y_te, label=""):
    clf = RandomForestClassifier(
        n_estimators=200, max_depth=6,
        min_samples_leaf=2, random_state=42, class_weight="balanced"
    )
    clf.fit(X_tr, y_tr)
    probs = clf.predict_proba(X_te)[:, 1]
    best_t, best_f1 = 0.35, 0
    for t in [i/20 for i in range(4, 14)]:
        preds = (probs > t).astype(int)
        f = f1_score(y_te, preds, zero_division=0)
        if f > best_f1:
            best_f1, best_t = f, t
    preds = (probs > best_t).astype(int)
    print(f"  [{label}] threshold={best_t:.2f}  F1={best_f1:.3f}  "
          f"P={precision_score(y_te, preds, zero_division=0):.3f}  "
          f"R={recall_score(y_te, preds, zero_division=0):.3f}  "
          f"AUC={roc_auc_score(y_te, probs):.3f}")
    return clf

print("Model comparison (RF, balanced class weights):")

X_tr_base, X_te_base, y_tr_base, y_te_base = train_test_split(
    features[FEATURE_COLS], features["Label_MSRIT"],
    test_size=0.25, random_state=42, stratify=features["Label_MSRIT"]
)
rf_base = eval_model(X_tr_base, y_tr_base, X_te_base, y_te_base, label="Baseline")

if len(NLP_FEATURE_COLS) > 0:
    rf_nlp = eval_model(X_train_nlp, y_train_nlp, X_test_nlp, y_test_nlp, label="+NLP ")
    model = rf_nlp
    X_train_bal, y_train_bal = X_train_nlp, y_train_nlp
    X_test_bal,  y_test_bal  = X_test_nlp, y_test_nlp
    FEATURE_COLS_FINAL = FEATURE_COLS_EXTENDED
else:
    model = rf_base
    X_train_bal, y_train_bal = X_tr_base, y_tr_base
    X_test_bal,  y_test_bal  = X_te_base, y_te_base
    FEATURE_COLS_FINAL = FEATURE_COLS
    print("  [Note] NLP columns not found — using baseline features.")
    print("  Re-run 02_feature_engineering.ipynb to generate NLP features.")


Model comparison (RF, balanced class weights):
  [Baseline] threshold=0.55  F1=0.667  P=0.667  R=0.667  AUC=0.937
  [+NLP ] threshold=0.45  F1=0.667  P=0.667  R=0.667  AUC=0.921


In [ ]:
try:
    from imblearn.over_sampling import SMOTE

    smote = SMOTE(random_state=42, k_neighbors=3)
    X_train_bal, y_train_bal = smote.fit_resample(X_train_nlp, y_train_nlp)

    print("After SMOTE:")
    print("Train size:", X_train_bal.shape[0])
    print("Class distribution:", pd.Series(y_train_bal).value_counts().to_dict())
    smote_available = True

except ImportError:
    print("imbalanced-learn not installed. Install with: pip install imbalanced-learn")
    print("Falling back to class_weight='balanced' only.")
    X_train_bal, y_train_bal = X_train_nlp, y_train_nlp
    smote_available = False

After SMOTE:
Train size: 124
Class distribution: {0: 62, 1: 62}


In [ ]:
%pip install xgboost

from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=6,
    min_samples_leaf=2,
    random_state=42,
    class_weight="balanced"
)
rf_model.fit(X_train_bal, y_train_bal)
print("Random Forest trained on", len(FEATURE_COLS_FINAL), "features")

try:
    xgb_model = XGBClassifier(
        n_estimators=200,
        max_depth=4,
        learning_rate=0.05,
        scale_pos_weight=y_train_bal.value_counts()[0] / y_train_bal.value_counts()[1],
        random_state=42,
        eval_metric="logloss",
        verbosity=0
    )
    xgb_model.fit(X_train_bal, y_train_bal)
    print("XGBoost trained")
    xgb_available = True
except Exception as e:
    print(f"XGBoost unavailable: {e}")
    xgb_available = False

model = rf_model


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
Random Forest trained on 29 features
XGBoost trained


In [ ]:
from sklearn.metrics import f1_score, precision_score, recall_score

probs_test = model.predict_proba(X_test_nlp)[:, 1]

thresholds = np.arange(0.20, 0.70, 0.05)
results = []
for t in thresholds:
    preds = (probs_test > t).astype(int)
    results.append({
        "Threshold": round(t, 2),
        "F1_gap":        round(f1_score(y_test_nlp, preds, zero_division=0), 3),
        "Precision_gap": round(precision_score(y_test_nlp, preds, zero_division=0), 3),
        "Recall_gap":    round(recall_score(y_test_nlp, preds, zero_division=0), 3),
    })

threshold_df = pd.DataFrame(results)
best_row = threshold_df.loc[threshold_df["F1_gap"].idxmax()]
best_threshold = best_row["Threshold"]

print("Random Forest threshold tuning:")
print(threshold_df.to_string(index=False))
print(f"Best threshold (RF): {best_threshold}")

if xgb_available:
    xgb_probs = xgb_model.predict_proba(X_test_nlp)[:, 1]
    xgb_results = []
    for t in thresholds:
        preds = (xgb_probs > t).astype(int)
        xgb_results.append({
            "Threshold": round(t, 2),
            "F1_gap":        round(f1_score(y_test_nlp, preds, zero_division=0), 3),
            "Precision_gap": round(precision_score(y_test_nlp, preds, zero_division=0), 3),
            "Recall_gap":    round(recall_score(y_test_nlp, preds, zero_division=0), 3),
        })
    xgb_threshold_df = pd.DataFrame(xgb_results)
    best_xgb_row = xgb_threshold_df.loc[xgb_threshold_df["F1_gap"].idxmax()]
    print(f"XGBoost best F1: {best_xgb_row['F1_gap']} at threshold {best_xgb_row['Threshold']}")
    print(f"RF best F1: {best_row['F1_gap']} at threshold {best_threshold}")
    winner = "XGBoost" if best_xgb_row["F1_gap"] > best_row["F1_gap"] else "Random Forest"
    print(f"Winner: {winner}")

Random Forest threshold tuning:
 Threshold  F1_gap  Precision_gap  Recall_gap
      0.20   0.500          0.400       0.667
      0.25   0.500          0.400       0.667
      0.30   0.571          0.500       0.667
      0.35   0.571          0.500       0.667
      0.40   0.571          0.500       0.667
      0.45   0.667          0.667       0.667
      0.50   0.667          0.667       0.667
      0.55   0.667          0.667       0.667
      0.60   0.400          0.500       0.333
      0.65   0.500          1.000       0.333
Best threshold (RF): 0.45
XGBoost best F1: 0.8 at threshold 0.3
RF best F1: 0.667 at threshold 0.45
Winner: XGBoost


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

y_pred = (probs_test > best_threshold).astype(int)

print("Classification Report:")
print(classification_report(y_test_nlp, y_pred, target_names=["Not a Gap", "Gap Skill"]))

print("Confusion Matrix:")
cm = confusion_matrix(y_test_nlp, y_pred)
cm_df = pd.DataFrame(cm,
    index=["Actual: Not Gap", "Actual: Gap"],
    columns=["Predicted: Not Gap", "Predicted: Gap"]
)
print(cm_df)

roc_auc = roc_auc_score(y_test_nlp, probs_test)
print(f"\nROC-AUC Score: {roc_auc:.3f}")
print("(ROC-AUC > 0.75 is good for imbalanced binary classification)")

Classification Report:
              precision    recall  f1-score   support

   Not a Gap       0.95      0.95      0.95        21
   Gap Skill       0.67      0.67      0.67         3

    accuracy                           0.92        24
   macro avg       0.81      0.81      0.81        24
weighted avg       0.92      0.92      0.92        24

Confusion Matrix:
                 Predicted: Not Gap  Predicted: Gap
Actual: Not Gap                  20               1
Actual: Gap                       1               2

ROC-AUC Score: 0.921
(ROC-AUC > 0.75 is good for imbalanced binary classification)


In [ ]:
importance = pd.DataFrame({
    "Feature": FEATURE_COLS_FINAL,
    "Importance": model.feature_importances_
}).sort_values("Importance", ascending=False)

print(importance.to_string(index=False))

                Feature  Importance
                   Freq    0.125934
    Company_Spread_Norm    0.125672
         Company_Spread    0.120940
             Volatility    0.098549
 Taught_Institute_Count    0.073924
            Trend_Slope    0.069895
  Semantic_Gap_Distance    0.063344
Institutional_Gap_Score    0.060487
              Cluster_4    0.046225
         Recency_Weight    0.045780
       NLP_Rarity_Score    0.037209
               Velocity    0.034944
    Semantic_Centrality    0.025652
              Cluster_7    0.020412
              Cluster_3    0.015966
          Upcoming_Flag    0.010343
      Domain_Web/Mobile    0.009813
           Domain_AI/ML    0.004612
              Cluster_1    0.002245
              Cluster_6    0.002002
            Domain_Data    0.001610
           Domain_Other    0.001275
   Domain_Systems/Infra    0.000969
              Cluster_0    0.000877
           Domain_Cloud    0.000810
              Cluster_5    0.000279
          Domain_DevOps    0

In [ ]:
importance_df = pd.DataFrame({
    "Feature":    FEATURE_COLS_FINAL,
    "Importance": model.feature_importances_
}).sort_values("Importance", ascending=False).reset_index(drop=True)

importance_df["Is_NLP"] = importance_df["Feature"].apply(
    lambda f: "★ NLP" if f in NLP_FEATURE_COLS else ""
)

print("Feature importance (top 20):")
print(importance_df.head(20).to_string(index=False))
print()
nlp_total = importance_df[importance_df["Is_NLP"]=="★ NLP"]["Importance"].sum()
print(f"Combined NLP feature importance: {nlp_total:.4f}  "
      f"({nlp_total/importance_df['Importance'].sum()*100:.1f}% of total)")


Feature importance (top 20):
                Feature  Importance Is_NLP
                   Freq    0.125934       
    Company_Spread_Norm    0.125672       
         Company_Spread    0.120940       
             Volatility    0.098549       
 Taught_Institute_Count    0.073924       
            Trend_Slope    0.069895       
  Semantic_Gap_Distance    0.063344  ★ NLP
Institutional_Gap_Score    0.060487       
              Cluster_4    0.046225  ★ NLP
         Recency_Weight    0.045780       
       NLP_Rarity_Score    0.037209  ★ NLP
               Velocity    0.034944       
    Semantic_Centrality    0.025652  ★ NLP
              Cluster_7    0.020412  ★ NLP
              Cluster_3    0.015966  ★ NLP
          Upcoming_Flag    0.010343       
      Domain_Web/Mobile    0.009813  ★ NLP
           Domain_AI/ML    0.004612  ★ NLP
              Cluster_1    0.002245  ★ NLP
              Cluster_6    0.002002  ★ NLP

Combined NLP feature importance: 0.2335  (23.4% of total)


In [19]:
from sklearn.model_selection import cross_val_predict, StratifiedKFold

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_probs = cross_val_predict(
    RandomForestClassifier(
        n_estimators=200, max_depth=6, min_samples_leaf=2,
        random_state=42, class_weight="balanced"
    ),
    X, y,
    cv=cv,
    method="predict_proba"
)[:, 1]

features["Recommendation_Prob_MSRIT"] = oof_probs
print("Out-of-fold probabilities computed — no data leakage")
print("Prob range:", oof_probs.min().round(3), "to", oof_probs.max().round(3))
print("Mean prob:", oof_probs.mean().round(3))

Out-of-fold probabilities computed — no data leakage
Prob range: 0.0 to 0.73
Mean prob: 0.144


In [20]:
industry = pd.read_csv("../outputs/cleaned_industry.csv")
skill_type = industry.groupby("Skill")["Skill_Type"].agg(lambda x: x.mode()[0]).reset_index()
skill_type.columns = ["Skill", "Trajectory"]

features = features.merge(skill_type, on="Skill", how="left")
features["Trajectory"] = features["Trajectory"].fillna("Unknown")

print("Trajectory distribution:")
print(features["Trajectory"].value_counts())

Trajectory distribution:
Trajectory
Transitional    51
Upcoming        26
Traditional     18
Name: count, dtype: int64


In [21]:
gap_skills = features[features["Is_Taught_MSRIT"] == 0].copy()

ranked_msrit = gap_skills.sort_values(
    "Recommendation_Prob_MSRIT", ascending=False
).reset_index(drop=True)

display_cols = [
    "Skill", "Trajectory", "Freq", "Velocity", "Volatility",
    "Company_Spread", "Recency_Weight", "Upcoming_Flag",
    "Demand_Score", "Label_MSRIT", "Recommendation_Prob_MSRIT"
]

print("Top 20 Recommended Skills for MSRIT Curriculum:")
print(ranked_msrit[display_cols].head(20).to_string(index=False))

Top 20 Recommended Skills for MSRIT Curriculum:
                 Skill   Trajectory  Freq  Velocity  Volatility  Company_Spread  Recency_Weight  Upcoming_Flag  Demand_Score  Label_MSRIT  Recommendation_Prob_MSRIT
              power bi Transitional    14 -2.000000    3.898718               9        0.357143              0        3.0514            1                   0.729900
                   aws  Traditional    29 -2.000000    4.494441              10        0.318966              0        6.8995            1                   0.694099
            typescript Transitional     9 -1.000000    2.489980               7        0.611111              0        2.1622            1                   0.679333
             terraform  Traditional    17 -0.333333    2.607681               9        0.602941              1        4.4406            1                   0.674645
                 ci/cd Transitional    10 -1.000000    2.549510               7        0.250000              0        2.3862   

In [22]:
ranked_msrit.to_csv("../outputs/ranked_skills_msrit.csv", index=False)
print("Saved ranked_skills_msrit.csv — shape:", ranked_msrit.shape)

features.to_csv("../outputs/model_output.csv", index=False)
print("Saved model_output.csv — shape:", features.shape)
print("Columns:", list(features.columns))

print("Top 10 gap skills for MSRIT:")
print(ranked_msrit[["Skill","Trajectory","Recommendation_Prob_MSRIT","Demand_Score","Company_Spread"]].head(10).to_string(index=False))

Saved ranked_skills_msrit.csv — shape: (52, 37)
Saved model_output.csv — shape: (95, 37)
Columns: ['Skill', 'Freq', 'Trend_Slope', 'Recency_Weight', 'Upcoming_Flag', 'Is_Taught_MSRIT', 'Taught_Institute_Count', 'Institutional_Gap_Score', 'Company_Spread', 'Company_Spread_Norm', 'Velocity', 'Volatility', 'Semantic_Centrality', 'Semantic_Gap_Distance', 'NLP_Rarity_Score', 'Skill_Cluster', 'Cluster_0', 'Cluster_1', 'Cluster_2', 'Cluster_3', 'Cluster_4', 'Cluster_5', 'Cluster_6', 'Cluster_7', 'Skill_Domain', 'Domain_AI/ML', 'Domain_Cloud', 'Domain_Data', 'Domain_DevOps', 'Domain_Other', 'Domain_Security', 'Domain_Systems/Infra', 'Domain_Web/Mobile', 'Demand_Score', 'Label_MSRIT', 'Recommendation_Prob_MSRIT', 'Trajectory']
Top 10 gap skills for MSRIT:
     Skill   Trajectory  Recommendation_Prob_MSRIT  Demand_Score  Company_Spread
  power bi Transitional                   0.729900        3.0514               9
       aws  Traditional                   0.694099        6.8995              10
